# Filling the gaps INE leaves in the empty and secondary dwellings

INE does not publish the empty/secondary split for every municipality. The 2011 census
gives it only for municipalities over 2,000 inhabitants, and the 2021 census classifies
dwellings by electricity use only for those over 1,000. That leaves **5,808
municipalities missing in 2011 and 4,992 in 2021** — the great majority of them, even
though they hold a minority of the country's dwellings.

`EmptyAndSecondaryDwellingsCensus(predict=True)` fills them in. This notebook explains
how, and then checks the result against a province where much of it can be verified.

In [ ]:
from social_ES import INE

# Everything is cached under `wd`; the first call downloads from INE, later ones read
# the local copy. Point this wherever you keep the data.
wd = "/mnt/data2/social_ES"

## Why this is more than guesswork

Filling in a missing number is usually a bad idea. Here it is defensible, because the
missing figures are **boxed in on two sides**:

* **The total each missing piece belongs to is published for the municipality itself.**
  2011 gives the non-main dwelling count for every municipality in the country, so what
  is unknown is how that count splits between secondary and empty, not how big it is.
  2021 gives every municipality's dwelling count.
* **The province total of every missing column is published too** — and the municipal
  counts of the totals above add up to their province *exactly*, checked across all 52
  provinces in both years. So the modelled figures of a province are not free: together
  they have to make up precisely the difference between the province's published total
  and the sum of its published municipalities.

On top of that, **2001 surveyed the split in every municipality in the country**. How a
place divided its non-main dwellings twenty years earlier is known even where 2011 is
silent, and it is a strong predictor of how it divides them now.

So a model is never asked to estimate a count outright — only to say **how a known
quantity divides**, which is a far smaller thing to ask of it.

## How it works

Gradient-boosted trees (`xgboost`) predict each missing share from what INE publishes
everywhere:

* the dwelling counts of all three censuses (total, main, non-main),
* the 2001 split, the only one that exists for every municipality,
* the 2021 census indicators describing what kind of place it is — age structure,
  tenure, household size, education, employment, and so on.

Each predicted share is then **scaled until its province adds up to the published figure
exactly**. The scaling is what turns a set of plausible guesses into a set that is
consistent with everything INE has actually said.

The `Predicted` column records, for every row, whether any of its figures were modelled.
Without `predict=True` nothing is filled in at all, and the gaps stay `NaN` — which is
what the censuses actually say.

## Lleida, as a test case

Lleida is a hard case and therefore a good one: 231 municipalities, most of them small
and mountainous, and INE publishes the split for very few of them.

In [ ]:
dwellings = INE.EmptyAndSecondaryDwellingsCensus(wd=wd, predict=True)

lleida = dwellings["Municipality"]
lleida = lleida[lleida["Municipality code"].str.startswith("25")]

(lleida.groupby(["Year", "Predicted"]).size().unstack(fill_value=0)
       .rename(columns={False: "published by INE", True: "modelled"}))

Predicted  published by INE  modelled
Year                                 
2001                    231         0
2011                     38       193
2021                     64       167

Only 38 of 231 in 2011, and 64 in 2021. 2001 is complete, which is why it needs no model
and is left exactly as published.

## Does it add up?

The first thing to check is the constraint the whole method rests on: summing the
municipalities of Lleida — published and modelled together — has to reproduce the figure
INE publishes for the province. For the columns the models actually write, it does, to
the dwelling.

In [ ]:
import pandas as pd

province = dwellings["Province"]
province = province[province["Province code"] == "25"].set_index("Year")


def reconcile(columns, years):
    """Municipal sums against INE's published province figure."""
    here = lleida[lleida["Year"].isin(years)]
    summed = here.groupby("Year")[columns].sum().stack()
    published = province.loc[list(years), columns].astype("float64").stack()
    return pd.DataFrame({"municipalities summed": summed.round().astype("int64"),
                         "INE's province figure": published.round().astype("int64"),
                         "difference": (summed - published).round().astype("int64")})


# 2011: the split of the non-main dwellings. Only the secondary ones are modelled —
# the empty ones are the remainder of a total that is already published.
reconcile(["Dwellings ~ Dwelling type:Secondary"], [2011])

                                          municipalities summed  INE's province figure  difference
Year                                                                                              
2011 Dwellings ~ Dwelling type:Secondary                  36496                  36496           0

In [ ]:
# 2021: the electricity classes. Three are modelled; regular use is the remainder.
reconcile(["Dwellings ~ Electricity use:Empty",
           "Dwellings ~ Electricity use:Very low consumption",
           "Dwellings ~ Electricity use:Sporadic use"], [2021])

                                                       municipalities summed  INE's province figure  difference
Year                                                                                                           
2021 Dwellings ~ Electricity use:Empty                                 40500                  40500           0
     Dwellings ~ Electricity use:Very low consumption                   9745                   9745           0
     Dwellings ~ Electricity use:Sporadic use                          24608                  24608           0

The harmonised `Comparable use` series is read off those columns and rounded to whole
dwellings per municipality, so summing 231 rounded numbers lands within about ten
dwellings of a total near 180,000. That is the rounding, not the model — and 2001, which
is published in full, is exact.

In [ ]:
reconcile(["Dwellings ~ Comparable use:Main",
           "Dwellings ~ Comparable use:Secondary",
           "Dwellings ~ Comparable use:Empty"], [2001, 2011, 2021])

                                           municipalities summed  INE's province figure  difference
Year                                                                                               
2001 Dwellings ~ Comparable use:Main                      128396                 128396           0
     Dwellings ~ Comparable use:Secondary                  35654                  35654           0
     Dwellings ~ Comparable use:Empty                      29626                  29626           0
2011 Dwellings ~ Comparable use:Main                      171174                 171180          -6
     Dwellings ~ Comparable use:Secondary                  36488                  36496          -8
     Dwellings ~ Comparable use:Empty                      37173                  37165           8
2021 Dwellings ~ Comparable use:Main                      181238                 181228          10
     Dwellings ~ Comparable use:Secondary                  34348                  34353          -5


## Is it any good?

Adding up is necessary but nowhere near sufficient. Spreading a province total evenly
across its municipalities would also add up, and would be worthless. The real question
is whether the model puts the dwellings in the **right** municipalities.

Lleida can answer that, because 38 of its municipalities in 2011 and 64 in 2021 *do*
have published figures. So: train on the rest of Spain with Lleida removed entirely,
predict its published municipalities as though they were missing, scale onto the total
the way the library does, and compare against what INE actually recorded.

The comparison is against giving every municipality the national rate — the obvious
thing to do without a model, and one that satisfies the province constraint just as well.

In [ ]:
import numpy as np
from xgboost import XGBRegressor

published = INE.EmptyAndSecondaryDwellingsCensus(wd=wd)["Municipality"]
features = INE._dwelling_use_features(wd, published)

for year, base_column, target, label in [
        (2011, "Dwellings ~ Dwelling type:Non-main",
               "Dwellings ~ Dwelling type:Secondary", "secondary dwellings"),
        (2021, "Dwellings", "Dwellings ~ Electricity use:Empty", "empty dwellings")]:

    block = published[published["Year"] == year].set_index("Municipality code")
    base = np.asarray(block[base_column], dtype=float)
    share = np.asarray((block[target] / block[base_column]).clip(0, 1))
    has_split = np.asarray(block[target].notna() & (block[base_column] > 0))
    in_lleida = np.asarray(block.index.str[:2] == "25")

    # Lleida is kept out of training entirely, so what the model says about it is said
    # about a province it has never seen.
    train, test = has_split & ~in_lleida, has_split & in_lleida
    X = features.reindex(block.index)

    model = XGBRegressor(n_estimators=500, max_depth=5, learning_rate=0.05,
                         subsample=0.85, colsample_bytree=0.85, min_child_weight=5,
                         reg_lambda=1.0, objective="reg:squarederror",
                         random_state=0, n_jobs=4)
    model.fit(X[train], share[train])
    predicted = np.clip(model.predict(X[test]), 0, 1)

    truth, weight = share[test] * base[test], base[test]
    total = truth.sum()
    national = float((share[train] * base[train]).sum() / base[train].sum())

    # Scaled onto the total exactly as the library does, the true total standing in for
    # the province figure INE publishes.
    ones = np.ones_like(predicted)
    modelled = INE._calibrate_shares(weight, predicted, ones, total) * weight
    flat = INE._calibrate_shares(weight, np.full_like(predicted, national), ones, total) * weight

    print(f"{year}  {label} — {test.sum()} municipalities of Lleida, "
          f"trained on {train.sum()} elsewhere")
    print(f"   model          mean error {np.abs(modelled - truth).mean():6.0f} dwellings"
          f"   correlation {np.corrcoef(modelled, truth)[0, 1]:.3f}")
    print(f"   national rate  mean error {np.abs(flat - truth).mean():6.0f} dwellings"
          f"   correlation {np.corrcoef(flat, truth)[0, 1]:.3f}")
    print(f"   totals match the province figure by construction: "
          f"{modelled.sum() - total:+.0f} dwellings\n")

2011  secondary dwellings — 38 municipalities of Lleida, trained on 2270 elsewhere
   model          mean error    107 dwellings   correlation 0.970
   national rate  mean error    205 dwellings   correlation 0.852
   totals match the province figure by construction: -0 dwellings

2021  empty dwellings — 64 municipalities of Lleida, trained on 3075 elsewhere
   model          mean error     81 dwellings   correlation 0.981
   national rate  mean error    156 dwellings   correlation 0.966
   totals match the province figure by construction: +0 dwellings



Roughly half the error of the national rate, and a correlation of 0.97–0.98 with what
INE recorded, in a province the model never saw.

**Two caveats worth keeping.** The error is a *mean over municipalities*, so a large one
can still be some way out. And Lleida's published municipalities are its larger ones,
while the municipalities actually being filled in are smaller — where a proportional
error is fewer dwellings but a harder prediction. Treat the modelled figures as good for
sums and for maps, and check `Predicted` before leaning on any single municipality.

The largest municipalities of Lleida in 2011 — the first is Lleida city itself, published;
most of the province is modelled:

In [ ]:
biggest = lleida[lleida["Year"] == 2011].nlargest(8, "Dwellings")
biggest[["Municipality code", "Predicted", "Dwellings",
         "Dwellings ~ Dwelling type:Non-main",
         "Dwellings ~ Dwelling type:Secondary",
         "Dwellings ~ Dwelling type:Empty"]].reset_index(drop=True)

  Municipality code  Predicted  Dwellings  Dwellings ~ Dwelling type:Non-main  Dwellings ~ Dwelling type:Secondary  Dwellings ~ Dwelling type:Empty
0             25120      False    66415.0                             10623.0                               3167.0                           7456.0
1             25217      False     8627.0                              2434.0                                316.0                           2118.0
2             25040      False     8261.0                              1957.0                                371.0                           1586.0
3             25203      False     7054.0                              1792.0                                673.0                           1119.0
4             25137      False     6577.0                              1225.0                                 41.0                           1184.0
5             25207      False     4927.0                              1247.0                                235

## On a map

`MapVariable` picks the `Predicted` column up on its own and puts a checkbox in the
legend. Unticking it drops the modelled municipalities from the map and from the table,
so what INE published can be seen on its own — and the tooltip marks the modelled ones.

In [ ]:
INE.MapVariable(lleida[lleida["Year"] == 2011],
                "Percentage of dwellings ~ Comparable use:Secondary",
                wd=wd, level="Municipality", boundaries_year=2011,
                title="Lleida — secondary dwellings, 2011")

Map written to /mnt/data2/social_ES/INE/Maps/lleida-secondary-dwellings-2011_municipalities_2011.html


'/mnt/data2/social_ES/INE/Maps/lleida-secondary-dwellings-2011_municipalities_2011.html'

## In short

* Nothing is filled in unless `predict=True` asks for it.
* What is filled in is consistent with every figure INE publishes: the municipal totals
  it belongs to, and the province totals it must sum to.
* `Predicted` says which rows were modelled, and both the maps and your own filtering
  can use it.
* Held out against a province the model never saw, it halves the error of the obvious
  alternative — but it remains an estimate, and a small municipality is estimated no
  better than a small municipality can be.

See the [`EmptyAndSecondaryDwellingsCensus` documentation](../docs/EmptyAndSecondaryDwellingsCensus.md)
for the full column reference, and [`get_ine.ipynb`](get_ine.ipynb) for the rest of the
library.